# Scenario Benchmark Runner

Run both tracking pipelines over identical videos and collect comparable runtime and telemetry statistics.

Launch Jupyter with the hardware variant you intend to test:

```bash
uv run --extra gpu --extra experiments jupyter lab
# or
uv run --extra cpu --extra experiments jupyter lab
```

Keep `DEVICE` aligned with the notebook kernel. This notebook measures throughput, reported unique IDs, and fragmentation proxies. Agreement between models is not accuracy; use the quality-analysis notebook with human ground truth for accuracy.

In [ ]:
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Run this notebook from the repository or notebooks directory: {ROOT}")

OUTPUT_DIR = ROOT / "outputs"
EXPERIMENT_DIR = OUTPUT_DIR / "experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "gpu" if torch.version.cuda is not None and torch.cuda.is_available() else "cpu"
if DEVICE == "gpu" and (torch.version.cuda is None or not torch.cuda.is_available()):
    raise RuntimeError("DEVICE='gpu' requires a CUDA-enabled notebook kernel")
if DEVICE == "cpu" and torch.version.cuda is not None:
    raise RuntimeError("DEVICE='cpu' requires the CPU-only notebook kernel")

print("Repository:", ROOT)
print("Device:", DEVICE)
print("Torch:", torch.__version__)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

## Configure scenarios and pipelines

Add representative videos for crowded scenes, occlusion/re-entry, camera motion, low light, and high resolution. Increase `repeats` after a one-run smoke test. The first run may include model downloads; use a warm-up run when comparing steady-state speed.

In [ ]:
SCENARIOS = [
    {
        "name": path.stem,
        "video": path,
        "sample_fps": 3,
        "repeats": 1,
        "line": None,  # Example for 4K: (0, 1080, 3839, 1080)
        "notes": "Classify as crowded, re-entry, camera-motion, low-light, or high-resolution",
    }
    for path in sorted((ROOT / "samples").glob("*.mp4"))
]

PIPELINES = [
    {
        "name": "rtdetr_osnet",
        "command": ["people-counter", "rtdetr-osnet"],
        "output_glob": "{stem}_telemetry_{device}_*.csv",
        "extra_args": ["--detector-model", "r18", "--detection-threshold", "0.6"],
    },
    {
        "name": "rfdetr_large_botsort",
        "command": ["people-counter", "rfdetr-botsort"],
        "output_glob": "{stem}_telemetry_rfdetr_large_botsort_{device}_*.csv",
        "extra_args": ["--detection-threshold", "0.6"],
    },
]

if not SCENARIOS:
    raise RuntimeError("No MP4 files found under samples/")
pd.DataFrame(SCENARIOS)

In [ ]:
def build_command(scenario, pipeline):
    command = [
        "uv", "run",
        "--extra", DEVICE,
        "--extra", "experiments",
        *pipeline["command"],
        str(scenario["video"]),
        "--device", DEVICE,
        "--sample-fps", str(scenario["sample_fps"]),
        *pipeline["extra_args"],
    ]
    if scenario.get("line") is not None:
        command.extend(["--line", *map(str, scenario["line"])])
    return command

for scenario in SCENARIOS:
    for pipeline in PIPELINES:
        print(scenario["name"], pipeline["name"])
        print("  ", " ".join(map(str, build_command(scenario, pipeline))))

## Run experiments

This cell executes each pipeline and locates the new timestamped telemetry file by comparing the output directory before and after each run. Failures include captured stdout/stderr for diagnosis.

In [ ]:
PROCESSING_PATTERN = re.compile(
    r"Processing time:\s*([0-9.]+)s\s*\(([0-9.]+) sampled FPS\)"
)

def run_benchmark(scenario, pipeline, repetition):
    video = Path(scenario["video"]).resolve()
    if not video.is_file():
        raise FileNotFoundError(video)
    pattern = pipeline["output_glob"].format(stem=video.stem, device=DEVICE)
    before = set(OUTPUT_DIR.glob(pattern))
    command = build_command({**scenario, "video": video}, pipeline)

    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    wall_seconds = time.perf_counter() - started
    if completed.returncode != 0:
        raise RuntimeError(
            f"Benchmark failed ({completed.returncode}): {' '.join(map(str, command))}\n"
            f"STDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}"
        )

    created = set(OUTPUT_DIR.glob(pattern)) - before
    if len(created) != 1:
        raise RuntimeError(f"Expected one new telemetry CSV for {pattern}, found {sorted(created)}")
    telemetry_path = created.pop()
    telemetry = pd.read_csv(telemetry_path)
    durations = pd.to_numeric(telemetry.get("duration_seconds", pd.Series(dtype=float)), errors="coerce")
    match = PROCESSING_PATTERN.search(completed.stdout)

    return {
        "scenario": scenario["name"],
        "pipeline": pipeline["name"],
        "repetition": repetition,
        "device": DEVICE,
        "sample_fps": scenario["sample_fps"],
        "command": " ".join(map(str, command)),
        "wall_seconds": wall_seconds,
        "processing_seconds": float(match.group(1)) if match else float("nan"),
        "processing_fps": float(match.group(2)) if match else float("nan"),
        "unique_people": len(telemetry),
        "zero_duration_tracks": int((durations == 0).sum()),
        "under_1s_tracks": int((durations < 1).sum()),
        "mean_track_duration": durations.mean(),
        "median_track_duration": durations.median(),
        "telemetry_path": str(telemetry_path.resolve()),
        "notes": scenario.get("notes", ""),
    }

records = []
for scenario in SCENARIOS:
    for pipeline in PIPELINES:
        for repetition in range(1, int(scenario["repeats"]) + 1):
            print(f"Running {scenario['name']} / {pipeline['name']} / repeat {repetition}")
            records.append(run_benchmark(scenario, pipeline, repetition))

BENCHMARK_RESULTS = pd.DataFrame(records)
BENCHMARK_RESULTS

In [ ]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
benchmark_path = EXPERIMENT_DIR / f"scenario_benchmark_{DEVICE}_{run_stamp}.csv"
BENCHMARK_RESULTS.to_csv(benchmark_path, index=False)
print("Saved:", benchmark_path)

plot_data = BENCHMARK_RESULTS.copy()
plot_data["short_track_ratio"] = (
    plot_data["under_1s_tracks"] / plot_data["unique_people"].replace(0, pd.NA)
)
metrics = [
    ("processing_seconds", "Processing seconds"),
    ("processing_fps", "Sampled FPS"),
    ("unique_people", "Reported unique people"),
    ("short_track_ratio", "Tracks shorter than 1 second"),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for axis, (metric, title) in zip(axes.flat, metrics):
    pivot = plot_data.pivot_table(index="scenario", columns="pipeline", values=metric, aggfunc="mean")
    pivot.plot(kind="bar", ax=axis, title=title)
    axis.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()